# Colab: Whisper e rembg com placa de vídeo
1. Menu **Ambiente de execução > Alterar o tipo > T4 GPU** (só na 1ª vez).
2. Na chave 🔑 (Secrets) à esquerda, crie `SENHA` com o conteúdo do arquivo `E:\Automacao\render-publico\.senha` e ligue o acesso do caderno (só na 1ª vez).
3. Escreva o NOME e o MODO abaixo e clique em **Ambiente de execução > Executar tudo**. No fim baixa `colab-NOME.zip`.

In [ ]:
NOME = 'teste'      # o nome que a central passar
MODO = 'whisper'    # 'whisper' ou 'rembg'

In [ ]:
import os, subprocess, zipfile, glob, json
from google.colab import userdata, files
os.environ['SENHA'] = userdata.get('SENHA')
url = f'https://github.com/JPSS1000/render-publico/releases/download/colab-{NOME}/pacote.bin'
subprocess.run(['wget', '-q', '-O', 'p.bin', url], check=True)
subprocess.run(['openssl','enc','-d','-aes-256-cbc','-pbkdf2','-iter','200000','-pass','env:SENHA','-in','p.bin','-out','p.zip'], check=True)
os.makedirs('ent', exist_ok=True); os.makedirs('sai', exist_ok=True)
zipfile.ZipFile('p.zip').extractall('ent'); os.remove('p.bin'); os.remove('p.zip')
print(len(glob.glob('ent/**/*', recursive=True)), 'arquivos')

In [ ]:
if MODO == 'whisper':
    subprocess.run('pip -q install faster-whisper', shell=True, check=True)
    from faster_whisper import WhisperModel
    m = WhisperModel('small', device='cuda', compute_type='float16')
    for a in sorted(glob.glob('ent/**/*', recursive=True)):
        if not a.lower().endswith(('.mp3', '.wav', '.m4a', '.mp4')): continue
        segs, _ = m.transcribe(a, language='pt', word_timestamps=True)
        pal, txt = [], []
        for s in segs:
            txt.append(s.text)
            pal += [{'palavra': w.word.strip(), 'ini': round(w.start, 3), 'fim': round(w.end, 3)} for w in s.words]
        base = os.path.join('sai', os.path.relpath(a, 'ent'))
        os.makedirs(os.path.dirname(base), exist_ok=True)
        json.dump(pal, open(base + '.palavras.json', 'w'), ensure_ascii=False)
        open(base + '.txt', 'w').write(''.join(txt).strip())
        print('ok', a, len(pal), 'palavras')

In [ ]:
if MODO == 'rembg':
    subprocess.run('pip -q install "rembg[gpu]" onnxruntime-gpu', shell=True, check=True)
    from rembg import remove, new_session
    from PIL import Image
    ses = new_session('isnet-general-use')
    for a in sorted(glob.glob('ent/**/*', recursive=True)):
        if not a.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')): continue
        out = os.path.join('sai', os.path.splitext(os.path.relpath(a, 'ent'))[0] + '.png')
        os.makedirs(os.path.dirname(out), exist_ok=True)
        remove(Image.open(a).convert('RGB'), session=ses).save(out)
        print('ok', out)

In [ ]:
z = f'colab-{NOME}.zip'
with zipfile.ZipFile(z, 'w') as f:
    for a in glob.glob('sai/**/*', recursive=True):
        if os.path.isfile(a): f.write(a, os.path.relpath(a, 'sai'))
files.download(z)